# Getting data from APIs with `requests`

**Goal:** fetch data from a web API, understand the response, handle errors and turn the result into a DataFrame.

We use two free APIs that need **no account or API key**:
- [PokeAPI](https://pokeapi.co): Pokémon data
- [Open-Meteo](https://open-meteo.com): weather forecasts

Install once (in your virtual environment): `pip install requests pandas`

## 1. What is an API?

An API lets one program ask another program for data over the internet. We send a **request** (a URL) and get a **response** back, usually as **JSON**. JSON looks exactly like the Python dicts and lists you already know.

In [ ]:
import requests
import pandas as pd

## 2. Our first request

In [ ]:
response = requests.get("https://pokeapi.co/api/v2/pokemon/25")
response

`<Response [200]>` means *OK*. Useful status codes: **200** OK, **404** not found, **429** too many requests, **500** server error.

In [ ]:
print(response.status_code)
print(response.ok)

## 3. The response is JSON → a Python dict

In [ ]:
pikachu = response.json()
print(type(pikachu))
print(list(pikachu.keys())[:10])

In [ ]:
print(pikachu['name'])
print(pikachu['weight'])
print(pikachu['height'])

### Nested data
`types` is a **list of dicts**, and each dict holds another dict. Here the dict and list skills from week 1 pay off.

In [ ]:
pikachu['types']

In [ ]:
pikachu['types'][0]['type']['name']

In [ ]:
type_names = [t['type']['name'] for t in pikachu['types']]
type_names

**Mini exercise:** get the names of all of Pikachu's abilities (look at `pikachu['abilities']`).

In [ ]:
# your code here

## 4. Query parameters
Many APIs accept **parameters** (`?key=value&key2=value2` at the end of the URL). Instead of building the URL by hand, pass a dict to `params=`.

In [ ]:
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 59.33,
    "longitude": 18.07,          # Stockholm
    "daily": "temperature_2m_max,precipitation_sum",
    "timezone": "Europe/Stockholm",
    "forecast_days": 7,
}
response = requests.get(url, params=params)
print(response.url)   # the full URL requests built for us
data = response.json()
data.keys()

In [ ]:
data['daily']

## 5. From JSON to DataFrame
The `daily` part is a dict of equally long lists, which is a perfect fit for `pd.DataFrame`.

In [ ]:
weather = pd.DataFrame(data['daily'])
weather['time'] = pd.to_datetime(weather['time'])
weather

## 6. When things go wrong
A good script never assumes the request worked.

In [ ]:
response = requests.get("https://pokeapi.co/api/v2/pokemon/not-a-real-pokemon")
print(response.status_code)

In [ ]:
# This cell fails ON PURPOSE: read the error message, it is the point of the demo\nresponse.raise_for_status()   # raises an HTTPError for 4xx/5xx

Wrap it in `try/except` and always set a `timeout`, otherwise your program can hang forever:

In [ ]:
def fetch_pokemon(name_or_id):
    url = f"https://pokeapi.co/api/v2/pokemon/{name_or_id}"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"Bad response for {name_or_id}: {e}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Network problem: {e}")
        return None
    return response.json()

print(fetch_pokemon("pikachu")['name'])
print(fetch_pokemon("not-a-real-pokemon"))

## 7. Many requests → one table
Loop over several ids, keep only the fields we need, and build a DataFrame from the resulting **list of dicts**.

In [ ]:
rows = []
for pokemon_id in range(1, 11):
    p = fetch_pokemon(pokemon_id)
    if p is None:
        continue
    rows.append({
        "id": p["id"],
        "name": p["name"],
        "height": p["height"],
        "weight": p["weight"],
        "types": ", ".join(t["type"]["name"] for t in p["types"]),
    })

df = pd.DataFrame(rows)
df

### Deeply nested JSON: `pd.json_normalize`
Flattens nested dicts into columns (`stats.0.base_stat` style).

In [ ]:
pd.json_normalize(pikachu, record_path="stats", meta=["name"])

## 8. Pagination
APIs rarely return everything at once. They return a page plus a link to the next one.

In [ ]:
page = requests.get("https://pokeapi.co/api/v2/pokemon", params={"limit": 20}).json()
print(page["count"], "pokemon in total")
print("next page:", page["next"])
len(page["results"])

In [ ]:
all_results = []
url = "https://pokeapi.co/api/v2/pokemon?limit=50"
pages = 0
while url and pages < 3:          # safety limit: only 3 pages in class
    page = requests.get(url, timeout=10).json()
    all_results.extend(page["results"])
    url = page["next"]
    pages += 1

pd.DataFrame(all_results).head()

## 9. Be a good API citizen
- **Save raw responses to disk** so you don't call again while testing.
- Respect **rate limits**; add `time.sleep(1)` in loops against stricter APIs.
- **Never hard-code API keys.** Put them in a `.env` file (added to `.gitignore`) and read them with `python-dotenv`; you will practise this with the stock pipeline in week 9.

In [ ]:
import json
from pathlib import Path

Path("data").mkdir(exist_ok=True)
with open("data/pikachu.json", "w") as f:
    json.dump(pikachu, f, indent=2)

with open("data/pikachu.json") as f:
    from_disk = json.load(f)
from_disk["name"]